# Embeddings 101

Quick tour of what embeddings are and what you can do with them.

- **Part 1**: word embeddings (GloVe) — print raw vectors, compare `dog` vs `cat`.
- **Part 2**: vector arithmetic — capital cities, and a gender-bias example worth being aware of.
- **Part 3**: document/sentence embeddings using a BERT-family model (`all-MiniLM-L6-v2`, a distilled BERT).

These models are trained by guessing a missing word from its context — e.g. "I'm going to France to learn ___" should predict *French*. Do that over billions of sentences and words used in similar contexts get pulled toward similar vectors — that's where the structure enabling addition/subtraction comes from.

Note: BERT's own embeddings are *contextual* (a word's vector depends on the sentence around it), so they don't support clean arithmetic like `king - man + woman`. For that we use GloVe, a classic *static* embedding model from the same era. We bring BERT back for Part 3, where contextual embeddings are exactly what you want.

In [1]:
%pip install -q gensim sentence-transformers scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


## Part 1: Word Embeddings

Each word maps to a fixed-length vector of numbers. Words with similar meanings end up with similar vectors.

In [2]:
import gensim.downloader as api

# 100-dimensional GloVe vectors trained on Wikipedia + Gigaword (~128MB download, cached after first run)
word_vectors = api.load("glove-wiki-gigaword-100")

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

[--------------------------------------------------] 0.0% 0.0/128.1MB downloaded

... (16,386 outputs left out)

[=================================================-] 100.0% 128.1/128.1MB downloaded

[=================================================-] 100.0% 128.1/128.1MB downloaded

[=================================================-] 100.0% 128.1/128.1MB downloaded

[==================================================] 100.0% 128.1/128.1MB downloaded

In [3]:
dog_vec = word_vectors["dog"]
cat_vec = word_vectors["cat"]

print(f"Vector for 'dog' has {len(dog_vec)} dimensions")
print("dog:", dog_vec[:10], "...")
print("cat:", cat_vec[:10], "...")

Vector for 'dog' has 100 dimensions
dog: [ 0.30817  0.30938  0.52803 -0.92543 -0.73671  0.63475  0.44197  0.10262
 -0.09142 -0.56607] ...
cat: [ 0.23088   0.28283   0.6318   -0.59411  -0.58599   0.63255   0.24402
 -0.14108   0.060815 -0.7898  ] ...


In [4]:
similarity = word_vectors.similarity("dog", "cat")
print(f"cosine similarity(dog, cat) = {similarity:.3f}")

# for contrast, an unrelated pair
print(f"cosine similarity(dog, umbrella) = {word_vectors.similarity('dog', 'umbrella'):.3f}")

cosine similarity(dog, cat) = 0.880
cosine similarity(dog, umbrella) = 0.234


## Part 2: Vector Arithmetic

Because these vectors encode meaning geometrically, you can add and subtract them. The classic example: `king - man + woman ≈ queen`.

In [5]:
result = word_vectors.most_similar(positive=["king", "woman"], negative=["man"], topn=5)
print("king - man + woman ≈")
for word, score in result:
    print(f"  {word:12s} {score:.3f}")

king - man + woman ≈
  queen        0.770
  monarch      0.684
  throne       0.676
  daughter     0.659
  princess     0.652


The same trick works for other relationships the model has learned, like country → capital city.

In [6]:
result = word_vectors.most_similar(positive=["paris", "italy"], negative=["france"], topn=5)
print("paris - france + italy ≈")
for word, score in result:
    print(f"  {word:12s} {score:.3f}")

paris - france + italy ≈
  rome         0.819
  milan        0.738
  naples       0.712
  venice       0.702
  turin        0.700


### A cautionary example

Embeddings learn whatever patterns exist in their training text — including social biases. A well-known illustration (see Bolukbasi et al., 2016, *"Man is to Computer Programmer as Woman is to Homemaker"*): `doctor - man + woman` has historically resolved to *nurse* rather than staying at *doctor*, reflecting bias baked into the training corpus rather than reality. Run the cell below to see whether that still shows up here — a good reminder to check for this in any production system built on embeddings (search ranking, resume screening, recommendations, etc.).

In [7]:
result = word_vectors.most_similar(positive=["doctor", "woman"], negative=["man"], topn=5)
print("doctor - man + woman ≈")
for word, score in result:
    print(f"  {word:12s} {score:.3f}")

doctor - man + woman ≈
  nurse        0.774
  physician    0.719
  doctors      0.682
  patient      0.675
  dentist      0.673


## Part 3: Document Embeddings

Whole sentences or paragraphs can be embedded too, using models built on BERT. Here we use `all-MiniLM-L6-v2`, a small, distilled BERT-family model tuned for producing good sentence-level vectors.

In [8]:
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

C:\Users\Kevin.Chovanec\AppData\Local\miniconda3\envs\embedding_guide\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Kevin.Chovanec\AppData\Local\miniconda3\envs\embedding_guide\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kevin.Chovanec\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8816.60it/s]

In [9]:
paragraph = (
    "Dogs are loyal companions that have lived alongside humans for thousands of years. "
    "They come in hundreds of breeds, ranging from tiny lapdogs to large working animals, "
    "and are known for their strong sense of smell and social bonds with their owners."
)

doc_vec = sentence_model.encode(paragraph)

print(f"Vector for paragraph has {len(doc_vec)} dimensions")
print(doc_vec[:10], "...")

Vector for paragraph has 384 dimensions
[-0.00551341 -0.04297237  0.08917701  0.10220753 -0.01573174  0.04899617
 -0.01494103 -0.0488873   0.03513964 -0.02800313] ...


Just like with words, we can compare documents by comparing their vectors — semantically similar text ends up close together, even without shared words.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

paragraph_about_cats = (
    "Cats are independent pets that have been domesticated for thousands of years. "
    "They are skilled hunters with sharp claws and keen night vision, and they often "
    "form close bonds with the people who care for them."
)
paragraph_about_finance = (
    "The central bank raised interest rates by a quarter point, citing persistent "
    "inflation pressures and a tight labor market. Investors reacted by selling off "
    "government bonds."
)

vecs = sentence_model.encode([paragraph, paragraph_about_cats, paragraph_about_finance])
sims = cosine_similarity(vecs)

print(f"dogs vs cats paragraph:    {sims[0][1]:.3f}")
print(f"dogs vs finance paragraph: {sims[0][2]:.3f}")

dogs vs cats paragraph:    0.489
dogs vs finance paragraph: -0.004
